In [4]:
# ============================================================
# D9 — Stage 4 Validation — Branch C
# 0. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import platform
import re
import sys

import pandas as pd


In [5]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D9"

DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População "
    "de Portugal — Ano de 1925"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

INPUT_REPRESENTATION = (
    "Complete deterministically normalised OCR structural Markdown"
)

EXPECTED_REFERENCE_RECORD_COUNT = 19

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

EXPECTED_SOURCE_SHA256 = (
    "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1"
)

OUTPUT_DIR = Path("outputs_D9_validation_C_non_evaluable")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Validation pathway: non-content-evaluable structured response")

Document: D9
Branch: C
Expected reference records: 19
Validation pathway: non-content-evaluable structured response


In [6]:
# ============================================================
# 2. Upload canonical D9 Branch C validation evidence
# ============================================================
# Required:
#   1) D9_reference_values.csv
#   2) D9_branch_C_structure_check.json
#   3) D9_branch_C_experiment_metadata.json
#   4) D9_branch_C_normalisation_check.json
#   5) D9_branch_C_experiment_summary.json
#
# A parsed extraction is intentionally NOT required because the
# preserved Branch C response was not content-evaluable.

print(
    "Upload:\n"
    "1. D9_reference_values.csv\n"
    "2. D9_branch_C_structure_check.json\n"
    "3. D9_branch_C_experiment_metadata.json\n"
    "4. D9_branch_C_normalisation_check.json\n"
    "5. D9_branch_C_experiment_summary.json"
)

uploaded = files.upload()

paths = [Path(name) for name in uploaded]

csv_paths = [
    p for p in paths
    if p.suffix.lower() == ".csv"
]

json_paths = [
    p for p in paths
    if p.suffix.lower() == ".json"
]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D9_reference_values.csv."
    )

REFERENCE_PATH = csv_paths[0]

STRUCTURE_CHECK_PATH = None
EXPERIMENT_METADATA_PATH = None
NORMALISATION_CHECK_PATH = None
EXPERIMENT_SUMMARY_PATH = None


def canonical_filename(path):
    return (
        path.name
        .casefold()
        .replace(" ", "_")
    )


# First pass: canonical filename patterns.
for path in json_paths:

    filename = canonical_filename(path)

    if "d9_branch_c_structure_check" in filename:
        STRUCTURE_CHECK_PATH = path
        continue

    if (
        "d9_branch_c_experiment_metadata" in filename
        and "_pre" not in filename
    ):
        EXPERIMENT_METADATA_PATH = path
        continue

    if (
        "d9_branch_c_normalisation_check" in filename
        or "d9_branch_c_normalization_check" in filename
    ):
        NORMALISATION_CHECK_PATH = path
        continue

    if "d9_branch_c_experiment_summary" in filename:
        EXPERIMENT_SUMMARY_PATH = path
        continue


# Second pass: content-based fallback.
for path in json_paths:

    with path.open("r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "parsed_extraction_created" in obj
        and "records_evaluable" in obj
        and "validation_status" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path
        continue

    if (
        NORMALISATION_CHECK_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_CHECK_PATH = path
        continue

    if (
        EXPERIMENT_METADATA_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "source_sha256" in obj
        and "raw_response_sha256" in obj
        and "structure_check_file" in obj
    ):
        EXPERIMENT_METADATA_PATH = path
        continue

    if (
        STRUCTURE_CHECK_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "records_evaluable" in obj
        and "valid_json" in obj
        and "validation_status" not in obj
    ):
        STRUCTURE_CHECK_PATH = path
        continue


required = {
    "structure check": STRUCTURE_CHECK_PATH,
    "experiment metadata": EXPERIMENT_METADATA_PATH,
    "normalisation check": NORMALISATION_CHECK_PATH,
    "experiment summary": EXPERIMENT_SUMMARY_PATH,
}

missing = [
    label
    for label, path in required.items()
    if path is None
]

if missing:
    raise ValueError(
        "Could not identify required files: "
        + ", ".join(missing)
    )


required_path_strings = [
    str(path)
    for path in required.values()
]

if len(required_path_strings) != len(set(required_path_strings)):
    raise ValueError(
        "The same JSON file was assigned to more than one artefact type."
    )


print("\nIdentified inputs:")
print("Reference:", REFERENCE_PATH.name)
print("Structure check:", STRUCTURE_CHECK_PATH.name)
print("Experiment metadata:", EXPERIMENT_METADATA_PATH.name)
print("Normalisation check:", NORMALISATION_CHECK_PATH.name)
print("Experiment summary:", EXPERIMENT_SUMMARY_PATH.name)

Upload:
1. D9_reference_values.csv
2. D9_branch_C_structure_check.json
3. D9_branch_C_experiment_metadata.json
4. D9_branch_C_normalisation_check.json
5. D9_branch_C_experiment_summary.json


Saving D9_branch_C_structure_check.json to D9_branch_C_structure_check.json
Saving D9_branch_C_normalisation_check.json to D9_branch_C_normalisation_check.json
Saving D9_branch_C_experiment_summary.json to D9_branch_C_experiment_summary.json
Saving D9_branch_C_experiment_metadata.json to D9_branch_C_experiment_metadata.json
Saving D9_reference_values.csv to D9_reference_values.csv

Identified inputs:
Reference: D9_reference_values.csv
Structure check: D9_branch_C_structure_check.json
Experiment metadata: D9_branch_C_experiment_metadata.json
Normalisation check: D9_branch_C_normalisation_check.json
Experiment summary: D9_branch_C_experiment_summary.json


In [7]:
# ============================================================
# 3. Hash input artefacts
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
STRUCTURE_CHECK_SHA256 = sha256_file(STRUCTURE_CHECK_PATH)
EXPERIMENT_METADATA_SHA256 = sha256_file(EXPERIMENT_METADATA_PATH)
NORMALISATION_CHECK_SHA256 = sha256_file(NORMALISATION_CHECK_PATH)
EXPERIMENT_SUMMARY_SHA256 = sha256_file(EXPERIMENT_SUMMARY_PATH)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Structure check SHA-256:", STRUCTURE_CHECK_SHA256)
print("Experiment metadata SHA-256:", EXPERIMENT_METADATA_SHA256)
print("Normalisation check SHA-256:", NORMALISATION_CHECK_SHA256)
print("Experiment summary SHA-256:", EXPERIMENT_SUMMARY_SHA256)

Reference SHA-256: 61a42ee9b636d83ee69bf9ddf90002f44d90496ebfe66e8682961feb5caa59cb
Structure check SHA-256: 6a45bc7f09ae4b6004b4659494151105cb2847b784e3acc5e3d11319f479264e
Experiment metadata SHA-256: fa05f4db6b9642e6799f65db1f5239bd8864713de8fbe2fad92eb550fd75fb11
Normalisation check SHA-256: 800a7341f40278743936a91fb180b6c64b22d0f99f272ee57987da3739bf5385
Experiment summary SHA-256: 5977ab0898ec0e69c6ffcb4505f578bf73bd592740b967e5fa87d51131b517ce


In [8]:
# ============================================================
# 4. Load and verify fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=True,
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)


def restore_reference_value(value):

    if value is None or pd.isna(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        try:
            number = float(text)
            return int(number) if number.is_integer() else number
        except ValueError:
            pass

    return value


reference_df["Value"] = reference_df["Value"].map(
    restore_reference_value
)

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

reference_record_count_valid = (
    len(reference_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


reference_semantic_checks = {
    "index_entry_values_null": bool(
        reference_df.loc[
            reference_df["Category"] == "Index entry",
            "Value"
        ].isna().all()
    ),

    "document_structure_values_null": bool(
        reference_df.loc[
            reference_df["Category"] == "Document structure",
            "Value"
        ].isna().all()
    ),

    "document_structure_periods_null": bool(
        reference_df.loc[
            reference_df["Category"] == "Document structure",
            "Reporting Period"
        ].isna().all()
    ),

    "portugal_area_period_null": bool(
        pd.isna(
            reference_df.loc[
                reference_df["Topic"] == "Portugal area",
                "Reporting Period"
            ].iloc[0]
        )
    ),
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)


print("Reference records:", len(reference_df))
print("Reference schema exact:", reference_schema_exact)
print("Reference record count valid:", reference_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Reference semantics valid:", reference_semantics_valid)

display(reference_df)

Reference records: 19
Reference schema exact: True
Reference record count valid: True
Reference category counts valid: True
Reference semantics valid: True


,Category,Topic,Description,Value,Unit,Reporting Period,Source Location
0,Publication metadata,Title,Publication title,Estatística do Movimento Fisiológico da Popula...,None,1925,PDF page 1 — Cover
1,Publication metadata,Reference year,Statistical reference year,1925,year,1925,PDF page 1 — Cover
2,Publication metadata,Publication year,Printed publication year,1929,year,1929,PDF page 1 — Imprint
3,Publication metadata,Publisher,Publication printer,Imprensa Nacional,None,1929,PDF page 1 — Imprint
4,Publication metadata,Institution,Responsible institution,Direcção Geral de Saúde — Portugal; Inspecção ...,None,1925,PDF page 1 — Cover
5,Index entry,Tabela I,Área e População recenseada (I-XII-1911 e I-XI...,None,None,1911 and 1920,PDF page 3 — Índice; printed page 1
6,Index entry,Tabela II,População recenseada (I-XII-1911 e I-XII-1920)...,None,None,1911 and 1920,PDF page 3 — Índice; printed pages 2 a 5
7,Index entry,Tabela III,"Casamentos, divórcios, nascimentos e óbitos, p...",None,None,None,PDF page 3 — Índice; printed pages 6 a 12
8,Index entry,Tabela XIV,"Óbitos, por idades e sexos, nos distritos",None,None,None,PDF page 3 — Índice; printed pages 70 e 71
9,Index entry,Tabela LVIII,"Taxas do movimento fisiológico, referidas à po...",None,None,None,PDF page 5 — Índice; printed pages 240 e 241


In [9]:
# ============================================================
# 5. Load Branch C execution evidence
# ============================================================

with STRUCTURE_CHECK_PATH.open(
    "r",
    encoding="utf-8-sig"
) as f:
    structure_check = json.load(f)

with EXPERIMENT_METADATA_PATH.open(
    "r",
    encoding="utf-8-sig"
) as f:
    experiment_metadata = json.load(f)

with NORMALISATION_CHECK_PATH.open(
    "r",
    encoding="utf-8-sig"
) as f:
    normalisation_check = json.load(f)

with EXPERIMENT_SUMMARY_PATH.open(
    "r",
    encoding="utf-8-sig"
) as f:
    experiment_summary = json.load(f)


for name, obj in {
    "structure_check": structure_check,
    "experiment_metadata": experiment_metadata,
    "normalisation_check": normalisation_check,
    "experiment_summary": experiment_summary,
}.items():

    if obj.get("document_id") != DOCUMENT_ID:
        raise AssertionError(
            f"{name} document_id does not match D9."
        )

    if obj.get("branch") != BRANCH:
        raise AssertionError(
            f"{name} branch does not match Branch C."
        )


source_hash_matches_stage_1 = (
    experiment_summary.get("source_sha256")
    == EXPECTED_SOURCE_SHA256
)

metadata_source_hash_matches_stage_1 = (
    experiment_metadata.get("source_sha256")
    == EXPECTED_SOURCE_SHA256
)


print("Source hash matches Stage 1:", source_hash_matches_stage_1)
print(
    "Metadata source hash matches Stage 1:",
    metadata_source_hash_matches_stage_1
)

print("\nExperiment summary:")
print(
    json.dumps(
        experiment_summary,
        indent=2,
        ensure_ascii=False
    )
)

Source hash matches Stage 1: True
Metadata source hash matches Stage 1: True

Experiment summary:
{
  "document_id": "D9",
  "document_name": "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D9 - EMovimentoFisiológico1925 (1).pdf",
  "source_sha256": "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised OCR structural Markdown",
  "representation_file": "D9_branch_C_normalised_markdown.md",
  "representation_sha256": "2e8b6810c818f27b7d598083749b236eccea4271fc64d0d6138d101c89688c7c",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "ocr_conversion_inherited_from_branch_B": true,
  "ocr_rerun_in_branch_C": false,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_s

In [10]:
# ============================================================
# 6. Confirm the non-content-evaluable Branch C outcome
# ============================================================
# This is the critical D9 Branch C validation pathway.
#
# No parsed extraction exists because the preserved model response
# was not valid JSON and did not contain an evaluable records list.
# These facts are preserved as the experimental result.

valid_json = experiment_summary.get("valid_json")
records_evaluable = experiment_summary.get("records_evaluable")
structure_valid = experiment_summary.get("structure_valid")
parsed_extraction_created = experiment_summary.get(
    "parsed_extraction_created"
)
parsed_extraction_sha256 = experiment_summary.get(
    "parsed_extraction_sha256"
)
accuracy_validation_completed = experiment_summary.get(
    "accuracy_validation_completed"
)

content_evaluable = bool(
    valid_json
    and records_evaluable
    and structure_valid
    and parsed_extraction_created
)

non_evaluable_reason = experiment_summary.get(
    "validation_status"
)


non_evaluable_checks = {
    "valid_json_is_false":
        valid_json is False,

    "records_evaluable_is_false":
        records_evaluable is False,

    "structure_valid_is_false":
        structure_valid is False,

    "parsed_extraction_created_is_false":
        parsed_extraction_created is False,

    "parsed_extraction_sha256_is_null":
        parsed_extraction_sha256 is None,

    "accuracy_validation_completed_is_false":
        accuracy_validation_completed is False,

    "content_evaluable_is_false":
        content_evaluable is False,
}

non_evaluable_status_confirmed = all(
    non_evaluable_checks.values()
)


print(
    json.dumps(
        non_evaluable_checks,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "Non-content-evaluable status confirmed:",
    non_evaluable_status_confirmed
)

print(
    "Reason:",
    non_evaluable_reason
)


if not non_evaluable_status_confirmed:
    raise AssertionError(
        "The supplied D9 Branch C evidence does not match the "
        "preserved non-content-evaluable execution."
    )

{
  "valid_json_is_false": true,
  "records_evaluable_is_false": true,
  "structure_valid_is_false": true,
  "parsed_extraction_created_is_false": true,
  "parsed_extraction_sha256_is_null": true,
  "accuracy_validation_completed_is_false": true,
  "content_evaluable_is_false": true
}
Non-content-evaluable status confirmed: True
Reason: Not content-evaluable because the preserved Branch C response does not contain an evaluable records structure


In [11]:
# ============================================================
# 7. Preserve Branch C normalisation-integrity diagnostics
# ============================================================

if normalisation_check.get("parent_branch") != PARENT_BRANCH:
    raise AssertionError(
        "Normalisation check parent branch does not match Branch B."
    )


representation_integrity = {
    "parent_branch":
        normalisation_check.get("parent_branch"),

    "parent_equivalence_passed":
        bool(
            normalisation_check.get(
                "parent_equivalence_passed",
                False
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_check.get(
                "normalisation_integrity_passed",
                False
            )
        ),

    "page_sequence_preserved":
        normalisation_check.get(
            "page_sequence_preserved"
        ),

    "deterministic_representation_verified":
        normalisation_check.get(
            "deterministic_representation_verified"
        ),

    "all_critical_markers_preserved":
        normalisation_check.get(
            "all_critical_markers_preserved"
        ),

    "rotation_evidence_preserved":
        normalisation_check.get(
            "rotation_evidence_preserved"
        ),

    "numeric_tokens_preserved":
        normalisation_check.get(
            "numeric_tokens_preserved"
        ),

    "complete_8_page_representation_retained":
        normalisation_check.get(
            "complete_8_page_representation_retained"
        ),

    "ocr_conversion_inherited_from_branch_B":
        normalisation_check.get(
            "ocr_conversion_inherited_from_branch_B"
        ),

    "ocr_rerun_applied":
        normalisation_check.get(
            "ocr_rerun_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_check.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_check.get(
            "semantic_rewriting_applied"
        ),

    "unit_conversion_applied":
        normalisation_check.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_check.get(
            "numeric_calculation_applied"
        ),

    "manual_reconstruction_applied":
        normalisation_check.get(
            "manual_reconstruction_applied"
        ),

    "manual_correction_applied":
        normalisation_check.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_check.get(
            "reference_values_used_for_transformation"
        ),

    "numeric_token_preservation":
        normalisation_check.get(
            "numeric_token_preservation"
        ),

    "critical_marker_status":
        normalisation_check.get(
            "critical_marker_status"
        ),

    "rotation_note_checks":
        normalisation_check.get(
            "rotation_note_checks"
        ),
}


print(
    json.dumps(
        representation_integrity,
        indent=2,
        ensure_ascii=False
    )
)


if not representation_integrity[
    "parent_equivalence_passed"
]:
    raise AssertionError(
        "Branch C parent-B equivalence did not pass."
    )

if not representation_integrity[
    "normalisation_integrity_passed"
]:
    raise AssertionError(
        "Branch C normalisation integrity did not pass."
    )

{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "all_critical_markers_preserved": true,
  "rotation_evidence_preserved": true,
  "numeric_tokens_preserved": true,
  "complete_8_page_representation_retained": true,
  "ocr_conversion_inherited_from_branch_B": true,
  "ocr_rerun_applied": false,
  "semantic_harmonisation_applied": false,
  "semantic_rewriting_applied": false,
  "unit_conversion_applied": false,
  "numeric_calculation_applied": false,
  "manual_reconstruction_applied": false,
  "manual_correction_applied": false,
  "reference_values_used_for_transformation": false,
  "numeric_token_preservation": {
    "four_digit_years": {
      "count_before": 24,
      "count_after": 24,
      "missing_token_count": 0,
      "added_token_count": 0,
      "passed": true
    },
    "comma_grouped_numbers": {
      "count_before": 1077,
      "count

In [12]:
# ============================================================
# 8. Create final D9 Branch C validation summary
# ============================================================
# IMPORTANT:
# Record- and field-level metrics are None, not zero.
#
# A zero score would imply that evaluable extracted records were
# compared and found incorrect. That did not occur here.

schema_validity = False

VALIDATION_METRICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records":
        int(len(reference_df)),

    "expected_reference_records":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "content_evaluable":
        False,

    "valid_json":
        False,

    "records_evaluable":
        False,

    "schema_validity":
        False,

    "structure_valid":
        False,

    "parsed_extraction_created":
        False,

    "parsed_extraction_sha256":
        None,

    "observed_record_count":
        None,

    "observed_category_counts":
        None,

    "scope_complete":
        False,

    "accuracy_validation_completed":
        False,

    "aligned_records":
        None,

    "fully_correct_records":
        None,

    "discrepant_records":
        None,

    "missing_records":
        None,

    "unsupported_extracted_records":
        None,

    "completeness":
        None,

    "missing_rate":
        None,

    "record_precision_exact":
        None,

    "record_recall_exact":
        None,

    "record_f1_exact":
        None,

    "overall_primary_field_accuracy":
        None,

    "field_accuracy_among_aligned":
        None,

    "category_metrics":
        None,

    "validation_status":
        (
            "Not content-evaluable because the preserved "
            "Branch C response does not contain an evaluable "
            "records structure"
        ),

    "non_evaluable_reason":
        non_evaluable_reason,

    "non_evaluable_checks":
        non_evaluable_checks,

    "branch_C_representation_integrity":
        representation_integrity,

    "reference_integrity_confirmation": {
        "reference_schema_exact":
            bool(reference_schema_exact),

        "reference_record_count_valid":
            bool(reference_record_count_valid),

        "reference_category_counts_valid":
            bool(reference_category_counts_valid),

        "reference_semantics_valid":
            bool(reference_semantics_valid),

        "reference_semantic_checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False,
    },

    "methodological_treatment": {
        "raw_response_preserved":
            bool(
                experiment_summary.get(
                    "raw_response_preserved"
                )
            ),

        "raw_response_regenerated":
            False,

        "manual_repair_applied":
            False,

        "invalid_structured_response_retained":
            True,

        "content_metrics_assigned_zero":
            False,

        "content_metrics_not_computed":
            True,

        "reason":
            (
                "The preserved response was not valid JSON and "
                "did not contain an evaluable records list. "
                "Accordingly, record alignment and field-level "
                "comparison were not performed."
            ),

        "comparison_rules_frozen_from_branch_A":
            True,
    },

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,

        "reference_sha256":
            REFERENCE_SHA256,

        "structure_check_file":
            STRUCTURE_CHECK_PATH.name,

        "structure_check_sha256":
            STRUCTURE_CHECK_SHA256,

        "experiment_metadata_file":
            EXPERIMENT_METADATA_PATH.name,

        "experiment_metadata_sha256":
            EXPERIMENT_METADATA_SHA256,

        "normalisation_check_file":
            NORMALISATION_CHECK_PATH.name,

        "normalisation_check_sha256":
            NORMALISATION_CHECK_SHA256,

        "experiment_summary_file":
            EXPERIMENT_SUMMARY_PATH.name,

        "experiment_summary_sha256":
            EXPERIMENT_SUMMARY_SHA256,

        "source_hash_matches_stage_1":
            bool(source_hash_matches_stage_1),

        "metadata_source_hash_matches_stage_1":
            bool(metadata_source_hash_matches_stage_1),

        "raw_response_sha256":
            experiment_summary.get(
                "raw_response_sha256"
            ),
    },

    "validation_timestamp":
        datetime.now(timezone.utc).isoformat(),
}


summary_table = pd.DataFrame([{
    "Document ID": DOCUMENT_ID,
    "Branch": BRANCH,
    "Reference Records": len(reference_df),
    "Content Evaluable": False,
    "Valid JSON": False,
    "Schema Validity": False,
    "Structure Valid": False,
    "Parsed Extraction Created": False,
    "Observed Records": None,
    "Scope Complete": False,
    "Accuracy Validation Completed": False,
    "Record F1 Exact": None,
    "Overall Primary Field Accuracy": None,
    "Normalisation Integrity Passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],
    "Validation Status":
        VALIDATION_METRICS[
            "validation_status"
        ],
}])


print(
    json.dumps(
        VALIDATION_METRICS,
        indent=2,
        ensure_ascii=False
    )
)

display(summary_table)

{
  "document_id": "D9",
  "document_name": "Estatística do Movimento Fisiológico da População de Portugal — Ano de 1925",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised OCR structural Markdown",
  "reference_records": 19,
  "expected_reference_records": 19,
  "content_evaluable": false,
  "valid_json": false,
  "records_evaluable": false,
  "schema_validity": false,
  "structure_valid": false,
  "parsed_extraction_created": false,
  "parsed_extraction_sha256": null,
  "observed_record_count": null,
  "observed_category_counts": null,
  "scope_complete": false,
  "accuracy_validation_completed": false,
  "aligned_records": null,
  "fully_correct_records": null,
  "discrepant_records": null,
  "missing_records": null,
  "unsupported_extracted_records": null,
  "completeness": null,
  "missing_rate": null,
  "record_precision_exact": null,
  "record_recall_exact": null,
  "record_f1_exact": null,
  "overal

,Document ID,Branch,Reference Records,Content Evaluable,Valid JSON,Schema Validity,Structure Valid,Parsed Extraction Created,Observed Records,Scope Complete,Accuracy Validation Completed,Record F1 Exact,Overall Primary Field Accuracy,Normalisation Integrity Passed,Validation Status
0,D9,C,19,False,False,False,False,False,None,False,False,None,None,True,Not content-evaluable because the preserved Br...


In [13]:
# ============================================================
# 9. Export D9 Branch C non-evaluable validation outputs
# ============================================================

SUMMARY_JSON_PATH = (
    OUTPUT_DIR
    / "D9_branch_C_validation_summary.json"
)

SUMMARY_CSV_PATH = (
    OUTPUT_DIR
    / "D9_branch_C_validation_summary.csv"
)

METADATA_JSON_PATH = (
    OUTPUT_DIR
    / "D9_branch_C_validation_metadata.json"
)

CONCLUSION_JSON_PATH = (
    OUTPUT_DIR
    / "D9_branch_C_validation_conclusion.json"
)


VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,

    "validation_type":
        "Post-extraction structural/non-evaluable outcome validation",

    "content_evaluable":
        False,

    "record_level_validation_performed":
        False,

    "field_level_validation_performed":
        False,

    "reference_modified":
        False,

    "raw_response_modified":
        False,

    "raw_response_regenerated":
        False,

    "manual_correction_applied":
        False,

    "comparison_rules_frozen_from_branch_A":
        True,

    "created_at":
        datetime.now(timezone.utc).isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),
}


VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "validation_status":
        VALIDATION_METRICS[
            "validation_status"
        ],

    "schema_valid":
        False,

    "content_evaluable":
        False,

    "validation_completed":
        True,

    "content_accuracy_validation_completed":
        False,

    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],

    "parent_B_equivalence_passed":
        representation_integrity[
            "parent_equivalence_passed"
        ],

    "record_f1_exact":
        None,

    "overall_primary_field_accuracy":
        None,

    "interpretation":
        (
            "Branch C preprocessing passed its deterministic "
            "normalisation-integrity checks, but the preserved "
            "LLM response was not valid JSON and did not contain "
            "an evaluable records structure. Record- and field-level "
            "metrics are therefore not computed."
        ),
}


SUMMARY_JSON_PATH.write_text(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

summary_table.to_csv(
    SUMMARY_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

METADATA_JSON_PATH.write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

CONCLUSION_JSON_PATH.write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)


print("Exported:")
print("-", SUMMARY_JSON_PATH)
print("-", SUMMARY_CSV_PATH)
print("-", METADATA_JSON_PATH)
print("-", CONCLUSION_JSON_PATH)

Exported:
- outputs_D9_validation_C_non_evaluable/D9_branch_C_validation_summary.json
- outputs_D9_validation_C_non_evaluable/D9_branch_C_validation_summary.csv
- outputs_D9_validation_C_non_evaluable/D9_branch_C_validation_metadata.json
- outputs_D9_validation_C_non_evaluable/D9_branch_C_validation_conclusion.json


In [14]:
# ============================================================
# 10. Final consistency checks and downloads
# ============================================================

assert reference_schema_exact
assert reference_record_count_valid
assert reference_category_counts_valid
assert reference_semantics_valid

assert source_hash_matches_stage_1
assert metadata_source_hash_matches_stage_1

assert non_evaluable_status_confirmed
assert content_evaluable is False

assert representation_integrity[
    "parent_equivalence_passed"
]

assert representation_integrity[
    "normalisation_integrity_passed"
]

assert VALIDATION_METRICS[
    "record_f1_exact"
] is None

assert VALIDATION_METRICS[
    "overall_primary_field_accuracy"
] is None

assert VALIDATION_METRICS[
    "parsed_extraction_created"
] is False

assert VALIDATION_METRICS[
    "schema_validity"
] is False

assert all(
    path.exists()
    for path in [
        SUMMARY_JSON_PATH,
        SUMMARY_CSV_PATH,
        METADATA_JSON_PATH,
        CONCLUSION_JSON_PATH,
    ]
)


print("D9 Validation C completed successfully.")
print("Content evaluable:", False)
print("Schema valid:", False)
print("Parsed extraction created:", False)
print(
    "Normalisation integrity passed:",
    representation_integrity[
        "normalisation_integrity_passed"
    ]
)
print("Exact F1: not computed")
print("Primary field accuracy: not computed")
print(
    "Validation status:",
    VALIDATION_METRICS[
        "validation_status"
    ]
)


for path in [
    SUMMARY_JSON_PATH,
    SUMMARY_CSV_PATH,
    METADATA_JSON_PATH,
    CONCLUSION_JSON_PATH,
]:
    files.download(path)

D9 Validation C completed successfully.
Content evaluable: False
Schema valid: False
Parsed extraction created: False
Normalisation integrity passed: True
Exact F1: not computed
Primary field accuracy: not computed
Validation status: Not content-evaluable because the preserved Branch C response does not contain an evaluable records structure


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>